In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import json
from pathlib import Path
from ast import literal_eval

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.tools import aggregate_results

In [ ]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def highlight_extrema(s, use_italics=False):  
    styles = []
    for v in s:      
        style = ''
        if not isinstance(v, (int, float, np.floating)):
            pass
        elif v == s.max():
            style = 'font-weight: bold;'
        elif v == s.min():
            style = 'font-weight: bold; font-style: italic;'
        styles.append(style)
    
    return styles

def sig3(x):
    if isinstance(x, (int, float, np.floating)):
        return f"{x:.3g}"
    return x

In [ ]:
### Load F1 data

metric_cols_f1 = ["theme f1 mean", "topic f1 mean", "concept f1 mean"]

res = pd.read_csv("./data/metrics.csv", sep=";")
res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])
res["suite"] = res.apply(get_suite, axis=1)

f1s = res[["model", "suite"] + metric_cols_f1]

f1s = f1s.rename(
    columns={
        "model": "Model",
        "suite": "Suite",
        "theme f1 mean": "Theme",
        "topic f1 mean": "Topic",
        "concept f1 mean": "Concept",
    }
)

f1s["Sum"] = f1s[f1s.columns[-3:]].sum(axis=1)

In [ ]:
### Baseline, max, min for each model and label
# In Use!
models = f1s["Model"].unique()
metric_cols = list(f1s.columns[-4:])

baselines = f1s["Suite"] == "0-non-expl"

max_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmax()
min_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmin()

best_per_label = pd.concat(
    [
        f1s[baselines],
        f1s.loc[max_values.values.reshape(-1)],
        f1s.loc[min_values.values.reshape(-1)],
    ]
).groupby(["Model", "Suite"]).max()

for model in models:
    per_model = best_per_label.loc[model]

    latex = per_model.style.apply(
        lambda col: highlight_extrema(
            col,
            True,
        )
    ).format(sig3).to_latex(convert_css=True)
    
    print("%" + model)
    print(latex)

In [ ]:
### Baseline, max, min for each model and label
# In Use!
models = f1s["model"].unique()
metric_cols = list(f1s.columns[-3:])

baselines = f1s["suite"] == "0-non-expl"

max_values = f1s[["model"] + metric_cols].groupby(by="model").idxmax()
min_values = f1s[["model"] + metric_cols].groupby(by="model").idxmin()

best_per_label = pd.concat(
    [
        f1s[baselines],
        f1s.loc[max_values.values.reshape(-1)],
        f1s.loc[min_values.values.reshape(-1)],
    ]
).groupby(["model", "suite"]).max()

for model in models:
    per_model = best_per_label.loc[model]

    latex = per_model.style.apply(
        lambda col: highlight_extrema(
            col,
            True,
        )
    ).format(sig3).to_latex(convert_css=True)
    
    print("%" + model)
    print(latex)